# 02 — Architecture U-Net

Objectifs de ce notebook :
- Comprendre le rôle de chaque composant du U-Net (encodeur, bottleneck, décodeur, skip connections)
- Comprendre pourquoi on prédit un **masque** plutôt que le spectrogramme propre directement
- Vérifier que le modèle tourne correctement : shapes d'entrée/sortie, nombre de paramètres
- Préparer le handoff vers le collègue qui écrit `train.py`

---
## 1. Rappel : ce que fait le modèle

Le modèle reçoit un **spectrogramme bruité** (une image 2D temps × fréquences) et doit produire le **spectrogramme propre** correspondant.

```
spectrogramme bruité (B, 1, 257, T)
          ↓
        U-Net
          ↓
spectrogramme propre prédit (B, 1, 257, T)
```

C'est un problème d'**image-to-image** : même taille en entrée et en sortie, mais le contenu change (on enlève le bruit).

Le U-Net est l'architecture parfaite pour ça : il a été inventé exactement pour ce type de problème (segmentation médicale, mais le principe est identique).

---
## 2. L'encodeur — "comprimer pour comprendre"

L'encodeur applique plusieurs fois la même séquence :

```
entrée (B, C_in, H, W)
    ↓ DoubleConv   → apprend des features, H et W restent identiques
    ↓ MaxPool(2×2) → divise H et W par 2, "zoome out"
sortie (B, C_out, H/2, W/2)
```

À chaque niveau, on **divise la résolution par 2** (MaxPool) mais on **double le nombre de canaux** (DoubleConv). Ça semble paradoxal, mais c'est intentionnel : on échange de la précision spatiale contre de la richesse sémantique.

Pour nos spectrogrammes (257 fréquences × 501 frames) avec base=32 :

| Niveau | Shape spatiale | Canaux | Ce que le réseau "voit" |
|--------|---------------|--------|-------------------------|
| Entrée | 257 × 501     | 1      | Magnitude brute pixel par pixel |
| Enc 1  | 257 × 501     | 32     | Textures locales (arêtes spectrales) |
| Enc 2  | 128 × 250     | 64     | Patterns de fréquences |
| Enc 3  | 64 × 125      | 128    | Structures sonores (formants, harmoniques) |
| Enc 4  | 32 × 62       | 256    | Caractéristiques globales du signal |
| Bottleneck | 16 × 31  | 512    | Empreinte compressée de tout le clip |

**Note sur les dimensions impaires** : 257 n'est pas une puissance de 2. MaxPool(2) arrondit à l'inférieur : 257 → 128. En remontant ×2, on obtiendrait 256 ≠ 257. On règle ça dans le décodeur avec `F.interpolate`.

---
## 3. Le bottleneck — "l'empreinte sonore"

Le bottleneck est le niveau le plus bas du U. C'est là que la représentation est la plus compressée : **16×31 = 496 positions spatiales** (contre 257×501 = 128,757 en entrée) avec 512 canaux.

À ce stade, le réseau n'a plus de détails fins — il a une vision globale du contenu sonore. C'est à partir de cette empreinte qu'il va devoir reconstruire le spectrogramme propre en remontant dans le décodeur.

C'est aussi pourquoi les skip connections sont indispensables : le bottleneck seul ne peut pas retrouver les détails perdus. Il a besoin des informations préservées par les skip connections.

---
## 4. Le décodeur + skip connections — "reconstruire avec les détails"

Le décodeur est le miroir de l'encodeur. Il applique à chaque niveau :

```
entrée (B, C, H/2, W/2)  ← vient du niveau inférieur
    ↓ F.interpolate(size=skip.shape[-2:])   → agrandit ×2 (fixe les dimensions impaires)
    ↓ cat([upsample, skip], dim=1)          → colle la skip connection en "largeur de canaux"
    ↓ DoubleConv                            → fusionne et raffine
sortie (B, C/2, H, W)
```

**La skip connection, c'est quoi concrètement ?**

Juste avant le MaxPool, on sauvegarde les features à pleine résolution (le `skip`). Ces features contiennent les détails fins que le MaxPool va effacer. En les injectant dans le décodeur via une concaténation, on "offre" au réseau une copie directe de l'information détaillée qu'il ne pouvait plus retrouver depuis le bottleneck.

```
Encodeur niveau 1 :   skip1 = features à 257×501  ──────────────────┐
Encodeur niveau 2 :   skip2 = features à 128×250  ─────────────┐    │
Encodeur niveau 3 :   skip3 = features à 64×125   ────────┐    │    │
Encodeur niveau 4 :   skip4 = features à 32×62    ───┐    │    │    │
                                                    │    │    │    │
Décodeur niveau 4 : upsample + cat(skip4) ──────────┘    │    │    │
Décodeur niveau 3 : upsample + cat(skip3) ───────────────┘    │    │
Décodeur niveau 2 : upsample + cat(skip2) ────────────────────┘    │
Décodeur niveau 1 : upsample + cat(skip1) ─────────────────────────┘
                           ↓
                       Conv 1×1
```

Le U-Net doit son nom exactement à ce schéma en forme de U.

---
## 5. Pourquoi un masque (IRM) et pas une prédiction directe ?

**Option A — prédiction directe** (naïve) :
```
sortie = ReLU(conv1x1(features))   # prédit la magnitude propre brute
```
Problème : le réseau peut prédire n'importe quelle valeur, y compris halluciner des fréquences qui n'existaient pas dans l'entrée.

**Option B — masque IRM** (notre choix) :
```
mask  = sigmoid(conv1x1(features))   # prédit un nombre entre 0 et 1 par bin
sortie = mask × noisy_mag             # atténue le spectrogramme bruité
```

C'est la bonne façon de penser le débruitage :
- Le réseau **filtre** des fréquences, il n'en invente pas.
- `mask ≈ 1` → cette fréquence est de la voix → on la conserve
- `mask ≈ 0` → cette fréquence est du bruit  → on la supprime
- La sortie est forcément dans **[0, noisy_mag]** → pas de magnitude négative possible

Le masque est aussi très **visualisable** : après l'entraînement, on peut l'afficher comme une heatmap et voir exactement ce que le réseau a appris à supprimer.

---
## 6. Setup — toujours exécuter en premier

In [ ]:
import os, sys
REPO_DIR = '/content/Filtre-Voix-DL'
assert os.path.exists(REPO_DIR), (
    f'{REPO_DIR} introuvable — exécute la cellule clone de main.ipynb '
    'pour cloner/mettre à jour le repo sur ce runtime Colab.'
)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

try:
    from IPython import get_ipython
    ipy = get_ipython()
    if ipy is not None:
        ipy.run_line_magic('load_ext', 'autoreload')
        ipy.run_line_magic('autoreload', '2')
except Exception:
    pass  # bug connu Colab Python 3.12, sans impact

print(f'sys.path OK — repo : {REPO_DIR}')

---
## 7. Instanciation et comptage des paramètres

In [ ]:
import torch
from src.model import UNet, count_parameters

model = UNet(base_channels=32)
n_params = count_parameters(model)

print(f'Modèle    : UNet(base_channels=32)')
print(f'Paramètres: {n_params:,}')   # attendu ~7,85M
print()

# Breakdown par composant
print('Détail par composant :')
for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f'  {name:12s} : {n:>10,} params')

---
## 8. Test avec un batch aléatoire (sans données réelles)

On vérifie d'abord que le modèle tourne sans erreur et que les shapes sont correctes, sans avoir besoin du Drive.

In [ ]:
# Simule un batch de 4 spectrogrammes bruités (magnitudes positives)
# Shape (B, C, H, W) = (batch=4, canaux=1, freq=257, temps=501)
x_fake = torch.randn(4, 1, 257, 501).abs()   # .abs() → valeurs ≥ 0 comme de vraies magnitudes

model.eval()   # désactive dropout/BN en mode entraînement (pas de bruit ajouté)
with torch.no_grad():
    out = model(x_fake)

print(f'Entrée  : {tuple(x_fake.shape)}')
print(f'Sortie  : {tuple(out.shape)}')
print()

# Vérifications fondamentales
assert out.shape == x_fake.shape, f'Shape mismatch : {out.shape} ≠ {x_fake.shape}'
assert (out >= 0).all(),          'La sortie contient des valeurs négatives !'
assert (out <= x_fake).all(),     'La sortie dépasse l\'entrée (masque > 1) !'

print('✓ Shape correcte')
print('✓ Sortie ≥ 0 (magnitudes valides)')
print('✓ Sortie ≤ entrée (le masque IRM atténue, il n\'amplifie pas)')

mask_mean = (out / (x_fake + 1e-8)).mean().item()
print(f'\nValeur moyenne du masque : {mask_mean:.3f}   (non entraîné → ~0.5 attendu)')

---
## 9. Test avec un vrai batch du dataset

Vérifie la compatibilité shapes entre ce que le dataset produit et ce que le modèle attend.

⚠️ Nécessite Drive monté et `data/` rempli (cf. `main.ipynb` → setup → cellule clone).

In [ ]:
from torch.utils.data import DataLoader
from src.dataset import PairedAudioDataset

ds     = PairedAudioDataset(return_spectrogram=True, pair_by='auto')
loader = DataLoader(ds, batch_size=4, shuffle=False, num_workers=0)
batch  = next(iter(loader))

print('Shapes du batch dataset :')
print(f'  noisy_mag  : {tuple(batch["noisy_mag"].shape)}')
print(f'  clean_mag  : {tuple(batch["clean_mag"].shape)}')
print(f'  noisy_phase: {tuple(batch["noisy_phase"].shape)}')
print()

model.eval()
with torch.no_grad():
    pred = model(batch['noisy_mag'])

print(f'Sortie modèle : {tuple(pred.shape)}')
print(f'Target (clean): {tuple(batch["clean_mag"].shape)}')

assert pred.shape == batch['clean_mag'].shape, 'Shape mismatch modèle / target !'
print('\n✓ La sortie du modèle a exactement la même shape que clean_mag')
print('  → train.py peut calculer la loss : criterion(pred, batch["clean_mag"])')

---
## 10. Visualisation d'un masque (non entraîné)

Affiche le masque produit par le modèle **avant** tout entraînement.
Il devrait être quasi-uniforme (~0.5 partout) puisque les poids sont initialisés aléatoirement.
Après entraînement, le masque devra montrer des zones claires (voix conservée) et sombres (bruit supprimé).

In [ ]:
import matplotlib.pyplot as plt
import librosa
import numpy as np
from src import config

# On récupère le masque interne en re-faisant la passe avant "à la main"
# (le modèle retourne mask * noisy_mag, mais on veut voir le masque seul)
model.eval()
noisy_input = batch['noisy_mag'][[0]]   # un seul exemple, shape (1, 1, F, T)

with torch.no_grad():
    # Reproduction du forward pour extraire le masque
    skip1, x = model.enc1(noisy_input)
    skip2, x = model.enc2(x)
    skip3, x = model.enc3(x)
    skip4, x = model.enc4(x)
    x = model.bottleneck(x)
    x = model.dec4(x, skip4)
    x = model.dec3(x, skip3)
    x = model.dec2(x, skip2)
    x = model.dec1(x, skip1)
    mask = torch.sigmoid(model.out_conv(x))   # (1, 1, F, T) ∈ ]0, 1[

noisy_np = noisy_input[0, 0].numpy()
mask_np  = mask[0, 0].numpy()
pred_np  = (mask * noisy_input)[0, 0].numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Spectrogramme bruité en entrée
librosa.display.specshow(librosa.amplitude_to_db(noisy_np, ref=np.max),
                         sr=config.SAMPLE_RATE, hop_length=config.HOP_LENGTH,
                         x_axis='time', y_axis='hz', ax=axes[0])
axes[0].set_title('Entrée noisy_mag (dB)')

# Le masque (non entraîné → ~0.5 uniforme)
im = axes[1].imshow(mask_np, aspect='auto', origin='lower',
                    vmin=0, vmax=1, cmap='RdYlGn')
axes[1].set_title('Masque IRM (non entraîné)')
axes[1].set_xlabel('Frames temporelles')
axes[1].set_ylabel('Fréquences')
plt.colorbar(im, ax=axes[1])

# La sortie : noisy_mag × masque
librosa.display.specshow(librosa.amplitude_to_db(pred_np + 1e-9, ref=np.max),
                         sr=config.SAMPLE_RATE, hop_length=config.HOP_LENGTH,
                         x_axis='time', y_axis='hz', ax=axes[2])
axes[2].set_title('Sortie prédite (non entraîné)')

plt.tight_layout()
plt.show()

print(f'Masque min={mask_np.min():.3f} | max={mask_np.max():.3f} | moy={mask_np.mean():.3f}')
print('(Non entraîné → valeurs proches de 0.5, sans structure visible)')

---
## 11. Ce qui reste à faire : `train.py`

Le modèle est prêt. L'interface pour le collègue qui écrit `train.py` est :

```python
from src.model import UNet, count_parameters
from src.dataset import PairedAudioDataset
from torch.utils.data import DataLoader
import torch.nn as nn

# Instanciation
model     = UNet(base_channels=32).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()   # ou L1Loss

# Boucle d'entraînement (schéma minimal)
for batch in dataloader:
    noisy = batch['noisy_mag'].to(device)   # (B, 1, 257, T)
    clean = batch['clean_mag'].to(device)   # (B, 1, 257, T)

    pred = model(noisy)                     # (B, 1, 257, T)
    loss = criterion(pred, clean)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
```

**Paramètre à passer (`base_channels`)** :
- `base=32` (~7,85M params) : recommandé, bon équilibre puissance/vitesse
- `base=16` (~1,97M params) : à essayer si l'entraînement est trop lent sur Colab free

**Pour reconstruire l'audio à l'inférence** (cf. `src/audio.py`) :
```python
from src import audio as A

with torch.no_grad():
    pred_mag   = model(noisy_mag)              # (1, 1, 257, T)
noisy_phase    = batch['noisy_phase'][0]       # (257, T)
audio_denoised = A.reconstruct(pred_mag[0, 0].numpy(), noisy_phase.numpy())
```